# QAOA — Quantum Approximate Optimization Algorithm

**Objective key:** `qaoa` &nbsp;·&nbsp; **Family:** Quantum &nbsp;·&nbsp; **Speed:** Compute-intensive

## Algorithm

QAOA solves the same **binary asset-selection QUBO** as `qubo_sa`,  
but uses a gate-model circuit instead of simulated annealing.

### Pipeline
1. Build the same QUBO matrix **Q** (shared with `qubo_sa`).
2. Map **Q** to an **Ising Hamiltonian**:
   - `h_i  = -Q[i,i]/2 − Σ_{j≠i} Q[i,j]/2`  (single-qubit Z)
   - `J_ij = Q[i,j]/2`  for `i < j`  (two-qubit ZZ)
3. Construct a **p-layer QAOA circuit**:
   - `|ψ₀⟩ = H^n|0…0⟩`  (uniform superposition)
   - Layer *l*: `U_C(γ_l) = exp(-i γ_l H_C)` then `U_B(β_l) = ⊗_i Rx(2β_l)`
4. Minimise `⟨ψ|H_C|ψ⟩` over angles **(γ, β)** via COBYLA.
5. Extract **highest-probability K-hot bitstring** → equal weight on selected assets.

### Execution paths

| Path | Backend | Cap | Notes |
|------|---------|-----|-------|
| Classical (default) | NumPy statevector | ≤12 assets | No IBM token needed |
| IBM Quantum | Qiskit Runtime — EstimatorV2 + SamplerV2 | ≤20 qubits | Token + `execution_kind: ibm_runtime` |

## Papers

- **Foundational:** Farhi, Goldstone & Gutmann (2014) — *A Quantum Approximate Optimization Algorithm*  
  https://arxiv.org/abs/1411.4028 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/1411.4028
- **Modern:** Zhou et al. (2020) — *QAOA: Performance, Mechanism, and Implementation on NISQ Devices*  
  https://arxiv.org/abs/1812.01041 &nbsp;·&nbsp; **PDF:** https://arxiv.org/pdf/1812.01041

## Assumptions

- `mu` and `Sigma` are annualised.
- Classical path is capped at `n ≤ 12` (statevector size `2^n`).
- `K=None` → auto (≈ `n // 3`). `p=2` layers, `n_restarts=3`, `seed=42`.
- Post-selection: equal weight on the K-hot bitstring with highest probability.

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api" / "app.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root — run from notebooks/objectives/ or repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 8  # must be <= 12 for classical statevector simulation
K = 3  # select 3 out of 8 assets
mu = rng.uniform(0.07, 0.22, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]
print(f"{n} assets (qubits), K={K}")
print("mu:", mu.round(4))

In [ ]:
# Classical numpy statevector path (no IBM token required)
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="qaoa",
    asset_names=asset_names,
    K=K,
    lambda_risk=1.0,
    gamma=8.0,
    n_restarts=3,
    seed=42,
)

active = [(name, w) for name, w in zip(asset_names, result.weights) if w > 1e-6]
print(f"\nObjective:       {result.objective}")
print(f"Selected assets: {len(active)} (target K={K})")
print(f"Expected return: {result.expected_return:.4f}")
print(f"Volatility:      {result.volatility:.4f}")
print(f"Sharpe ratio:    {result.sharpe_ratio:.4f}")
print("\nSelected assets (equal weight):")
for name, w in active:
    print(f"  {name}: {w:.4f}")

In [ ]:
# Show the QUBO -> Ising mapping for transparency
from methods.qubo_sa import _build_qubo_matrix
from methods.qaoa import _qubo_to_ising

Q = _build_qubo_matrix(mu, Sigma, K=K, lambda_risk=1.0, gamma=8.0)
h, J = _qubo_to_ising(Q)
print(f"Ising h (single-qubit Z): {h.round(4)}")
print(f"Ising J (ZZ pairs, {len(J)} terms):")
for (i, j), val in list(J.items())[:5]:
    print(f"  J[{i},{j}] = {val:.4f}")
if len(J) > 5:
    print(f"  ... ({len(J)} total ZZ terms)")

In [ ]:
# Compare QAOA vs QUBO-SA on the same data
qubo_sa_result = run_optimization(
    returns=mu, covariance=Sigma, objective="qubo_sa",
    asset_names=asset_names, K=K, lambda_risk=1.0, gamma=8.0, seed=42,
)
print("Sharpe comparison:")
print(f"  QAOA (classical):  {result.sharpe_ratio:.4f}")
print(f"  QUBO-SA:           {qubo_sa_result.sharpe_ratio:.4f}")
print("\nNote: QAOA and QUBO-SA solve the same problem; results may differ")
print("due to different solvers (circuit vs annealing), angle initialisation, and seed.")

In [ ]:
# IBM Quantum Runtime path (informational — requires IBM token)
print("IBM Quantum Runtime path for QAOA:")
print("  Requires: configured IBM token (POST /api/config/ibm-quantum)")
print("  Payload:  execution_kind='ibm_runtime', objective='qaoa'")
print("  Ansatz:   p-layer QAOA (cost = ZZ/Z rotations, mixer = Rx per qubit)")
print("  Angle optimisation: EstimatorV2 (energy expectation) + COBYLA")
print("  Bitstring extraction: SamplerV2 (shots=2048) on final optimal circuit")
print(f"  Max qubits (IBM cap): 20  (MAX_IBM_QUBITS in methods/qaoa.py)")
print(f"  Classical cap:        12  (MAX_SIM_QUBITS — statevector 2^n)")
print("\nOptional params in run payload:")
print("  p            — circuit depth (default 2)")
print("  K            — cardinality (default n//3)")
print("  lambda_risk  — risk aversion in QUBO (default 1.0)")
print("  gamma        — cardinality penalty (default 8.0)")